# TASK-003-05 Python最小基盤のカーネル確認

003の最小package（zizai-runtime 0.0.0.dev0）を、VS Codeで選択した既存Pythonカーネルから確認します。対応手順はリポジトリの `tests/manual/TASK-003-05-python.md` を参照してください。

初期化は通常セルのimportだけです。pip・path変更・独自実行・個別コネクタ処理はこのNotebookにありません。出力は未保存です。共有するのは下の診断JSONと対象版だけにし、環境確認セルの出力やNotebook全体は共有前に消去してください。

「すべて実行」だけでは再起動確認になりません。手順に沿ってセルを選んで実行してください。


## 1. カーネルとcwdのローカル確認

この出力は導入先を特定するためのローカル確認用です。個人の絶対pathが表示されるので、共有結果へ含めないでください。Notebookの保存先とカーネルcwdが同じとは仮定しません。


In [ ]:
import os
import sys
print("Python:", sys.version.split()[0])
print("選択カーネルのPython（ローカル確認用）:", sys.executable)
print("カーネルcwd（ローカル確認用）:", os.getcwd())


## 2. 診断の定義

このセルは標準ライブラリだけで診断関数を定義します。絶対pathや環境変数の中身を共有JSONへ出力しません。製品APIや実行Engineではありません。


In [ ]:
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import sys
from pathlib import Path


def _task003_report(phase):
    before = (os.getcwd(), dict(os.environ), tuple(sys.path))
    names = ("zizai", "zizai.core", "zizai.connectors")
    modules = [importlib.import_module(name) for name in names]
    after = (os.getcwd(), dict(os.environ), tuple(sys.path))
    distribution = importlib.metadata.distribution("zizai-runtime")
    package_paths = (
        "zizai/__init__.py", "zizai/core/__init__.py", "zizai/connectors/__init__.py",
    )
    matches = []
    files = []
    for module, expected_path in zip(modules, package_paths):
        actual = Path(module.__file__).resolve()
        expected = Path(distribution.locate_file(expected_path)).resolve()
        matches.append(actual == expected)
        files.append({
            "module": module.__name__,
            "sha256": hashlib.sha256(actual.read_bytes()).hexdigest(),
        })
    try:
        ipykernel_version = importlib.metadata.version("ipykernel")
    except importlib.metadata.PackageNotFoundError:
        ipykernel_version = None
    return {
        "schema_version": 1,
        "scope": "minimal_package_import",
        "phase": phase,
        "runtime": {
            "python": platform.python_version(),
            "implementation": platform.python_implementation(),
            "platform": sys.platform,
            "machine": platform.machine(),
            "ipykernel_distribution": ipykernel_version,
        },
        "distribution": {
            "name": distribution.metadata["Name"],
            "version": distribution.version,
            "requires_python": distribution.metadata.get("Requires-Python"),
            "runtime_dependency_count": len(distribution.requires or []),
        },
        "checks": {
            "imports_match_distribution": all(matches),
            "import_preserved_process_state": before == after,
        },
        "files": files,
    }


## 3. 導入後・再起動前の確認

手順どおり選択カーネルのPythonへwheelを手動導入し、必要なら標準再起動してから実行します。imports_match_distribution と import_preserved_process_state がtrueであることを確認してください。このセルは再起動前のセッション識別用変数も作ります。


In [ ]:
report_before_restart = _task003_report("before_restart")
print(json.dumps(report_before_restart, ensure_ascii=False, indent=2))
_task003_session_marker = object()


## 4. 標準カーネル再起動後の確認

VS Codeの標準操作でカーネルを再起動し、セル1（環境確認）とセル2（診断定義）だけを再実行してから、次のセルを実行します。セル3は再実行しません。

previous_session_marker_absent がtrueなら前セッションの変数はありません。同じセッションで順に実行するとfalseになるので、再起動済みとは判定できません。変数が消えたことは、カーネル再起動と変数削除を独立に識別する証明ではありません。標準の再起動操作を行った事実も記録してください。

ファイルhashはディスク上のファイルの値です。キャッシュ済みコードが新版に替わった証拠や、個別アクションの動作確認には使いません。


In [ ]:
report_after_restart = _task003_report("after_restart")
report_after_restart["checks"]["previous_session_marker_absent"] = (
    "_task003_session_marker" not in globals()
)
print(json.dumps(report_after_restart, ensure_ascii=False, indent=2))
